In [1]:
import numpy as np
import pandas as pd
import Soil_Carbon_Model as md 
import os
import openpyxl
from pandas import *

In [2]:
dt = 1 / 24
nyears = 1000
ndays = nyears * 365
nsim = ndays * 24
nnpp = ndays * 24

#Rainfall from Marked Poisson
rain = md.poisson_process(n = int(nsim), alpha = 11.88/1000, lbd = (0.22 * dt))

tsim = np.linspace(1, nnpp, int(nnpp)) # Hour

days = np.arange(365)
annual_avg_npp = 560

shape = np.sin((np.pi*days)/365)                    

daily_npp = annual_avg_npp * (shape / np.sum(shape))
hourly_npp = np.repeat(daily_npp, 24)

npp = np.tile(hourly_npp,nyears)

In [3]:
print("Maximum Value is:", max(npp))
print("Average value is:", np.mean(npp))

Maximum Value is: 2.4099814458967175
Average value is: 1.5342465753424661


In [4]:
total_rainfall = rain.sum()                  # total rainfall in 'm'
print(f"Total rainfall over 450 years = {total_rainfall:.3f} meters")

RPY = (total_rainfall / nyears)*1000         # average rainfall per year in 'mm'
print(f"Average rainfall per year = {RPY:.3f} mm/year")

PET = 884.04
AI = PET/RPY
print(AI)

Total rainfall over 450 years = 960.109 meters
Average rainfall per year = 960.109 mm/year
0.9207707245169077


In [5]:
# Compute TI_DI and TI_Texture

KA_final = 9.0132
n_final = 1.7444
R_max = 3.8                   

def compute_TI(AI):
    AI = np.asarray(AI, dtype=float)
    numerator   = AI**n_final
    denominator = KA_final**n_final + AI**n_final
    norm_factor = R_max**n_final / (KA_final**n_final + R_max**n_final)
    TI = (numerator / denominator) / norm_factor

    return np.minimum(TI, 1.0)

# ------------------------------------
# TI FROM SAND (SIGMOID)
# ------------------------------------
sand = 25.0   # PGL sand (%)
s50 = 25.0
k   = 0.08

def TI_from_sand(sand):
    return 1.0 / (1.0 + np.exp(-k * (sand - s50)))

TI_sand = TI_from_sand(sand)
TI_AI = compute_TI(AI)
TI = (TI_sand + TI_AI)/2

print("TI_AI =", TI_AI)
print("TI_sand =", TI_sand)
print("TI Total =", TI)

TI_AI = 0.10115700536088837
TI_sand = 0.5
TI Total = 0.30057850268044417


In [6]:
#Hydraulic parameterization for dynamic porosity -------------------------------
alpha1 = 580  #hydraulic conductivity shape parameter
beta1 = 6     #hydraulic conductivity shape parameter
alpha2 = 580  #porosity shape parameter
beta2 = 6     #porosity shape parameter
kstex = 0.15   #m/d,
kstot = 1     #m/d
phitex = 0.471  
phitot = 0.60  

In [7]:
# Simulation for 2150 years at dynamic porosity
sim = md.run_julia(rain, npp, kstex, kstot, phitex, phitot, alpha1, beta1, alpha2, beta2, TI)
sim_year = sim.pivot_table(values=['s', 'phi', 'ksstr', 'P', 'DOC', 'A', 'M', 'B',
                                   'Fa', 'Fpa','Flb','Fl', 'Flm', 'Fpl', 'Fbm','Fld', 'Fal',
                                   'lk','npp_', 'L','Fi', 'Flpom', 'decomp'],
                           index='Years')

print(sim_year)

100%|██████████████████████████████████████████████████████████████████████| 8760000/8760000 [16:18<00:00, 8954.69it/s]


                 A           B         DOC        Fa       Fal       Fbm  \
Years                                                                      
1       914.564049   73.782552   21.497682  0.126613  0.003476  0.268019   
2       937.691575   22.599459   20.797171  0.129578  0.002234  0.082094   
3      1023.226922    6.972852   20.422946  0.141653  0.001129  0.025329   
4      1094.987955    2.147857   20.630971  0.150492  0.000443  0.007802   
5      1140.647992    0.668252   21.737640  0.158293  0.000154  0.002427   
...            ...         ...         ...       ...       ...       ...   
996    1806.545386  185.917927  211.871930  0.251611  0.008282  0.675270   
997    1806.537425  190.490851  218.105923  0.248024  0.008186  0.691869   
998    1806.442079  207.253333  209.647313  0.244343  0.008138  0.752751   
999    1806.359934  206.132053  210.046593  0.251251  0.008364  0.748690   
1000   1806.237288  201.138400  206.353273  0.247618  0.008221  0.730549   

           

In [9]:
sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Puget & Lal\PGL.csv", index= True, header = True)
sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Results\PGL.csv", index= True, header = True)